In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

aerial_cactus_identification_path = kagglehub.competition_download('aerial-cactus-identification')

print('Data source import complete.')


In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

In [ ]:
from pathlib import Path
from fastai.vision.all import *
import torch

In [ ]:
!cp /kaggle/input/competitions/aerial-cactus-identification/train.zip /kaggle/working/train_modified.zip

In [ ]:
!cp /kaggle/input/competitions/aerial-cactus-identification/test.zip /kaggle/working/test_modified.zip

In [ ]:
data_folder = Path("/kaggle/working")

In [ ]:
import os
import zipfile

dataset_path = '/kaggle/working'

print(f"Checking contents of {dataset_path}:")

has_zip = False
for item in os.listdir(dataset_path):
    item_path = os.path.join(dataset_path, item)
    print(f"- {item}")
    if item.endswith('.zip'):
        has_zip = True
        print(f"  Found zip file: {item}. Extracting...")
        try:
            with zipfile.ZipFile(item_path, 'r') as zip_ref:
                zip_ref.extractall(dataset_path)
            print(f"  Extracted {item} to {dataset_path}")
        except zipfile.BadZipFile:
            print(f"  Warning: Could not extract {item}, it might be a corrupted or encrypted zip file.")


if has_zip:
    print("\nDirectory structure after extraction:")
else:
    print("\nNo zip files found. Directory structure:")

# Now print the full directory structure
for dirpath, dirnames, filenames in os.walk(dataset_path):
    relative_path = os.path.relpath(dirpath, dataset_path)
    if relative_path == '.':
        depth = 0
    else:
        depth = relative_path.count(os.sep) + 1

    indent = ' ' * 4 * depth
    print(f"{indent}├── {os.path.basename(dirpath)}/")

In [ ]:
train_df = pd.read_csv(f"{aerial_cactus_identification_path}/train.csv")
test_df = pd.read_csv(f"{aerial_cactus_identification_path}/sample_submission.csv")

In [ ]:
import zipfile

# Define paths to the zip files
train_zip_path = f"{aerial_cactus_identification_path}/train.zip"
test_zip_path = f"{aerial_cactus_identification_path}/test.zip"

# Define the destination directory for extraction
extract_to_path = data_folder

print(f"Extracting {train_zip_path} to {extract_to_path}...")
with zipfile.ZipFile(train_zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to_path)
print("Train images extracted.")

print(f"Extracting {test_zip_path} to {extract_to_path}...")
with zipfile.ZipFile(test_zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to_path)
print("Test images extracted.")

print("Image extraction complete.")

In [ ]:
# DataBlock setup for fastai v2

# Item transforms (applied to individual images)
item_tfms = Resize(128)

# Batch transforms (applied to a batch of images for augmentation and normalization)
# This replaces the fastai v1 `get_transforms` and `normalize`
batch_tfms = [
    *aug_transforms(
        do_flip=True,
        flip_vert=True,
        max_rotate=10.0,
        max_zoom=1.1,
        max_lighting=0.2,
        max_warp=0.2,
        p_affine=0.75,
        p_lighting=0.75
    ),
    Normalize.from_stats(*imagenet_stats)
]

# Define the DataBlock
dblock = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    # `get_x` extracts the image path from the DataFrame row
    get_x=lambda r: data_folder/f"train/{r['id']}",
    # `get_y` extracts the label from the DataFrame row
    get_y=ColReader('has_cactus'),
    # Split the data into training and validation sets
    splitter=RandomSplitter(valid_pct=0.01, seed=42),
    item_tfms=item_tfms,
    batch_tfms=batch_tfms
)

# Create DataLoaders for training and validation
dls = dblock.dataloaders(train_df, bs=64)

# Create a separate DataLoader for the test set (for inference)
# The `test_dl` method requires a list of image paths or a DataFrame.
# We'll provide an ordered list of paths to match the submission format later.
test_paths = [data_folder/f"test/{img_id}" for img_id in test_df['id']]
test_dl = dls.test_dl(test_paths)

# `dls` now holds your training and validation data, and `test_dl` holds your test data.

In [ ]:
learn = cnn_learner(dls, models.densenet161, metrics=[error_rate, accuracy])

In [ ]:
#learn.lr_find()
#learn.recorder.plot()

In [ ]:
lr = 3e-02
learn.fit_one_cycle(5, slice(lr))

In [ ]:
#learn.unfreeze()
#learn.lr_find()
#learn.recorder.plot()

In [ ]:
#learn.fit_one_cycle(1, slice(1e-06))


In [ ]:
#interp = ClassificationInterpretation.from_learner(learn)
#interp.plot_top_losses(9, figsize=(7,6))

In [ ]:
preds,_ = learn.get_preds(dl=test_dl)

In [ ]:
test_df.has_cactus = preds.numpy()[:, 1]

In [ ]:
test_df.to_csv('submission.csv', index=False)

In [ ]:
test_df.head()